Imports

In [ ]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset , DataLoader , random_split
import torch.nn as nn
import torch
import torch.optim as optim

Uploading data

In [ ]:
data_frame = pd.read_csv('teriage_dataset.csv') #Current path

Preprocessing

In [ ]:
data_frame_proccessd = data_frame.copy()
categorical_cols = ['sex', 'arrival_mode', 'mental_status', 'complaint_category']

for col in categorical_cols:
    unique_values = data_frame_proccessd[col].unique()
    mapping = {val: idx for idx, val in enumerate(unique_values)}
    data_frame_proccessd[col] = data_frame_proccessd[col].map(mapping)
    print(f"Mapping برای {col}: {mapping}")

feature_cols = [col for col in data_frame_proccessd.columns if col != 'target_level']
X = data_frame_proccessd[feature_cols].values.astype(np.float32)
y = data_frame_proccessd['target_level'].values.astype(np.int64)
y = y - 1

#Standardization
mean = X.mean(axis=0)
std = X.std(axis=0)
std[std == 0] = 1.0

X = (X - mean) / std

#To tensor
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

Preparing data

class TriageDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


full_dataset = TriageDataset(X_tensor, y_tensor)


train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # برای reproducibility
)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

Model

class TriageNet(nn.Module):
  def __init__(self, input_dim=13, num_classes=5):
      super(TriageNet, self).__init__()

      self.network = nn.Sequential(
       nn.Linear(input_dim , 512),
       nn.ReLU(),
       nn.Dropout(0.5),

       nn.Linear(512 , 256),
       nn.ReLU(),
       nn.Dropout(0.5),

       nn.Linear(256 , 128),
       nn.ReLU(),
       nn.Dropout(0.5),
       
       nn.Linear(128 , 64),
       nn.ReLU(),
       nn.Dropout(0.5),

       nn.Linear(64 , 32),
       nn.ReLU(),
       nn.Dropout(0.5),
    
       nn.Linear(32 , num_classes)
      )

  def forward(self , x):
    return self.network(x)

model = TriageNet(input_dim= X_tensor.shape[1] , num_classes=5)
print(model)


sample_X, sample_y = next(iter(train_loader))
with torch.no_grad():
    output = model(sample_X)
    print(output.shape)
    print(output[0])

Engine

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def calculate_accuracy(loader, model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    return 100 * correct / total


num_epochs = 50
train_losses = []
val_accuracies = []
train_accuracies = []

print("Starting train loop...\n")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        # Forward
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)
    
   
    train_acc = calculate_accuracy(train_loader, model)
    val_acc = calculate_accuracy(val_loader, model)
    
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("\nThe End!")
print(f"Best Validation Accuracy: {max(val_accuracies):.2f}%")

Validation

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)


final_acc = 100 * (all_preds == all_labels).sum() / len(all_labels)
print(f"Validation Accuracy Finally: {final_acc:.2f}%")


num_classes = 5
conf_matrix = np.zeros((num_classes, num_classes), dtype=int)

for true, pred in zip(all_labels, all_preds):
    conf_matrix[true][pred] += 1

print("\nConfusion Matrix (row = cloumn = Predict):")
print("     ", "  ".join([f"Pred{i}" for i in range(num_classes)]))
for i in range(num_classes):
    print(f"True{i}", conf_matrix[i])


print("\nReport of per class:")
for i in range(num_classes):
    TP = conf_matrix[i][i]
    FP = conf_matrix[:, i].sum() - TP
    FN = conf_matrix[i, :].sum() - TP
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"class {i+1} (target_level={i+1}): Precision={precision:.2f} | Recall={recall:.2f} | F1={f1:.2f}")